<a href="https://colab.research.google.com/github/Moyohor/Machine_Learning_Projects/blob/main/Copy_of_Phishing_URL_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORT AND EXTRACT THE FILE

In [ ]:
!unzip '/content/sample_data/Phishing_Legitimate_full.csv.zip'

Archive:  /content/sample_data/Phishing_Legitimate_full.csv.zip
  inflating: Phishing_Legitimate_full.csv  


IMPORT ALL REQUIRED LIBRARIES

In [ ]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score


Dataset Shape: (10000, 50) (Rows, Columns)

First 3 rows of the data:
   id  NumDots  SubdomainLevel  PathLevel  UrlLength  NumDash  \
0   1        3               1          5         72        0   
1   2        3               1          3        144        0   
2   3        3               1          2         58        0   

   NumDashInHostname  AtSymbol  TildeSymbol  NumUnderscore  ...  \
0                  0         0            0              0  ...   
1                  0         0            0              2  ...   
2                  0         0            0              0  ...   

   IframeOrFrame  MissingTitle  ImagesOnlyInForm  SubdomainLevelRT  \
0              0             0                 1                 1   
1              0             0                 0                 1   
2              0             0                 0                 1   

   UrlLengthRT  PctExtResourceUrlsRT  AbnormalExtFormActionR  \
0            0                     1                   

ESTABLISH STATISTICS OF THE DATASET

In [ ]:
# 1. Load the phishing dataset
# (Adjust the filename to match your exact uploaded CSV)
df = pd.read_csv('/content/Phishing_Legitimate_full.csv')

# 2. Print shape and columns to see if features are pre-extracted or raw text
print(f"Dataset Shape: {df.shape} (Rows, Columns)")
print("\nFirst 3 rows of the data:")
print(df.head(3))

# 3. Check the balance of safe links vs phishing links
# (Look at your columns to find the label name, e.g., 'Result', 'Label', or 'status')
for col in df.columns:
    if col.lower() in ['label', 'result', 'status', 'target']:
        print(f"\nClass distribution in column '{col}':")
        print(df[col].value_counts())
        break


Dataset Shape: (10000, 50) (Rows, Columns)

First 3 rows of the data:
   id  NumDots  SubdomainLevel  PathLevel  UrlLength  NumDash  \
0   1        3               1          5         72        0   
1   2        3               1          3        144        0   
2   3        3               1          2         58        0   

   NumDashInHostname  AtSymbol  TildeSymbol  NumUnderscore  ...  \
0                  0         0            0              0  ...   
1                  0         0            0              2  ...   
2                  0         0            0              0  ...   

   IframeOrFrame  MissingTitle  ImagesOnlyInForm  SubdomainLevelRT  \
0              0             0                 1                 1   
1              0             0                 0                 1   
2              0             0                 0                 1   

   UrlLengthRT  PctExtResourceUrlsRT  AbnormalExtFormActionR  \
0            0                     1                   

TRAIN AND TEST THE MODELS LOGISTIC REGRESSION, RANDOM FOREST,AND XGBOOST


In [ ]:

# 1. Drop the 'id' column if it exists, and separate features from target
X = df.drop(['id', 'CLASS_LABEL'], axis=1, errors='ignore')
y = df['CLASS_LABEL']

# 2. Split the dataset (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# 3. Scale the numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data cleaning, splitting, and scaling complete!")
print(f"Train size: {X_train_scaled.shape}, Test size: {X_test_scaled.shape}")


Data cleaning, splitting, and scaling complete!
Train size: (8000, 48), Test size: (2000, 48)


In [ ]:

# 1. Initialize the models
lr_model = LogisticRegression(max_iter=1000, random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)

models = {
    "Logistic Regression": lr_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

# 2. Train and evaluate
for name, model in models.items():
    print(f"\n================ {name} ================")
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    probabilities = model.predict_proba(X_test_scaled)[:, 1]

    # Print metrics
    print(classification_report(y_test, predictions, target_names=['Safe', 'Phishing']))
    print(f"ROC-AUC Score: {roc_auc_score(y_test, probabilities):.4f}")



================ Logistic Regression ================
              precision    recall  f1-score   support

        Safe       0.96      0.94      0.95      1000
    Phishing       0.94      0.96      0.95      1000

    accuracy                           0.95      2000
   macro avg       0.95      0.95      0.95      2000
weighted avg       0.95      0.95      0.95      2000

ROC-AUC Score: 0.9869

================ Random Forest ================
              precision    recall  f1-score   support

        Safe       0.99      0.99      0.99      1000
    Phishing       0.99      0.98      0.99      1000

    accuracy                           0.99      2000
   macro avg       0.99      0.99      0.99      2000
weighted avg       0.99      0.99      0.99      2000

ROC-AUC Score: 0.9989

================ XGBoost ================
              precision    recall  f1-score   support

        Safe       0.99      0.99      0.99      1000
    Phishing       0.99      0.99      0.99   

SAVE THE MODEL FOR DEPLOYMENT

In [ ]:

# 1. Save your best model (change 'xgb_model' to 'rf_model' if Random Forest won)
with open('phishing_detector_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

# 2. Save your scaler
with open('phishing_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Project 3 Complete! Your model and scaler files are saved and ready to export.")


Project 3 Complete! Your model and scaler files are saved and ready to export.
